# MTKD4SER: Multi-Teacher Knowledge Distillation for Speech Emotion Recognition

This notebook provides a complete pipeline to train and evaluate the **MTKD4SER** model on **Google Colab (Free T4 GPU / 15GB VRAM)**.

### Key Highlights:
- **Language-Aware Dynamic Routing**: Oracle Language routing, Uniform, Cosine similarity, and Confidence routing.
- **VRAM Optimized**: Dynamic dual-teacher setup (English + French) avoiding Finnish teacher duplication, safely running on T4 GPU.
- **Hugging Face Arrow Datasets**: Zero-copy memory mapping loading in <0.2s without Google Drive I/O bottlenecks.
- **Speaker-Independent Splits**: 0% speaker leakage across Train, Dev, and Test sets.

## 1. Verify GPU Hardware
Colab provides Nvidia T4 (15 GB VRAM). Ensure CUDA is available.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (Recommended)
Mount Google Drive to persist trained checkpoints (`/content/checkpoints`) and access large dataset archives across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Install Dependencies
Replace the repository URL below with your GitHub repository.

In [ ]:
# Clone the repository (or upload project folder)
!git clone https://github.com/<your-username>/mtkd4ser_phase1.git
%cd mtkd4ser_phase1

# Install project dependencies
!pip install -q transformers datasets torchaudio soundfile librosa scikit-learn tqdm tensorboard accelerate pyarrow wandb


### Hugging Face Authentication (Optional but Recommended)
Setting  avoids rate limiting and download warnings on Google Colab.
You can store  in Colab Secrets (the key icon 🔑 on the left sidebar) or set it directly below.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN successfully loaded from Colab Secrets.")
except Exception:
    # Alternatively, uncomment and paste token directly:
    # os.environ["HF_TOKEN"] = "hf_..."
    print("No HF_TOKEN detected in Colab Secrets (public models still download, but rate limits may apply).")

### Weights & Biases (W&B) Authentication (Recommended)
Logging to [Weights & Biases](https://wandb.ai) gives you live remote loss curves, teacher routing weight distributions, confusion matrices, and automatic cloud model artifact versioning.

Store your `WANDB_API_KEY` in Colab Secrets (the 🔑 icon on the left sidebar) or log in interactively.

In [ ]:
import os
try:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
    if wandb_key:
        import wandb
        wandb.login(key=wandb_key)
        print("✓ W&B successfully authenticated from Colab Secrets.")
    else:
        print("ℹ No WANDB_API_KEY detected in Colab Secrets. You can log in interactively:")
        import wandb
        wandb.login()
except Exception as e:
    print(f"ℹ Interactive login fallback: {e}")
    # import wandb
    # wandb.login()


## 4. Dataset Preparation

### Option A: Extract Pre-packaged Archive from Google Drive (Fastest)
If you uploaded `data_ser_phase1.tar.gz` to your Google Drive:

In [ ]:
# Example extracting from Drive into the project root
!unzip -q /content/drive/MyDrive/hf_data.zip -d ./

# Verify that data/hf_datasets exists
import os
print("Data exists:", os.path.exists("data"))
if os.path.exists("data/hf_datasets"):
    print("Hugging Face Arrow datasets found:", os.listdir("data/hf_datasets"))

### Quick Dataset Verification
Test that Hugging Face Arrow datasets load in zero-copy mode:

In [ ]:
from data import build_multilingual, iemocap, cafe

label2id, id2label, ds = build_multilingual(session_en=1, session_fr=1)
print("Multilingual Dataset structure:")
print(ds)
print("Emotion labels:", label2id)

## 5. Train Monolingual Teacher Models
Before running MTKD, train fine-tuned (FT) monolingual teachers for English and French.

In [ ]:
# 1. English Monolingual Teacher (IEMOCAP Session 1 Fold)
!python main.py --PARADIGM FT --LANGUAGE EN --LINGUALITY Monolingual \
    --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 \
    --TRAINING 1 --SAVE_DIR /content/checkpoints \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


In [ ]:
# 2. French Monolingual Teacher (CaFE Fold 1)
!python main.py --PARADIGM FT --LANGUAGE FR --LINGUALITY Monolingual \
    --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 8 --LEARNING_RATE 3e-5 \
    --TRAINING 1 --SAVE_DIR /content/checkpoints \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


## 6. Multi-Teacher Knowledge Distillation (MTKD)
Train student model using both English and French teachers.

### Recommended: Oracle Language-Aware Routing
Uses sample `language_id` (0=EN, 2=FR) to dynamically route loss to the native language teacher.

In [ ]:
!python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
    --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
    --TRAINING 1 --ROUTING language \
    --TEACHER_EN_CKPT /content/checkpoints/FT_Monolingual_EN_S1.pth \
    --TEACHER_FR_CKPT /content/checkpoints/FT_Monolingual_FR_S1.pth \
    --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


### Routing Ablations
Compare with baseline routing strategies:

In [ ]:
# Uniform baseline (50/50 teacher weights)
# !python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
#     --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
#     --TRAINING 1 --ROUTING uniform \
#     --TEACHER_EN_CKPT /content/checkpoints/FT_Monolingual_EN_S1.pth \
#     --TEACHER_FR_CKPT /content/checkpoints/FT_Monolingual_FR_S1.pth \
#     --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints

# Cosine similarity routing (Original paper approach)
# !python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
#     --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
#     --TRAINING 1 --ROUTING cosine \
#     --TEACHER_EN_CKPT /content/checkpoints/FT_Monolingual_EN_S1.pth \
#     --TEACHER_FR_CKPT /content/checkpoints/FT_Monolingual_FR_S1.pth \
#     --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints

# Confidence routing (Max softmax probability)
# !python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
#     --SESSION 1 --N_EPOCHS 12 --BATCH_SIZE 6 --LEARNING_RATE 3e-5 \
#     --TRAINING 1 --ROUTING confidence \
#     --TEACHER_EN_CKPT /content/checkpoints/FT_Monolingual_EN_S1.pth \
#     --TEACHER_FR_CKPT /content/checkpoints/FT_Monolingual_FR_S1.pth \
#     --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints

## 7. Model Evaluation & Confusion Matrix
Evaluate any saved checkpoint on the test set by setting `--TRAINING 0`.

In [ ]:
!python main.py --PARADIGM MTKD --LANGUAGE EN --LINGUALITY Multilingual \
    --SESSION 1 --TRAINING 0 --ROUTING language \
    --TEACHER_EN_CKPT /content/checkpoints/FT_Monolingual_EN_S1.pth \
    --TEACHER_FR_CKPT /content/checkpoints/FT_Monolingual_FR_S1.pth \
    --INCLUDE_FI 0 --SAVE_DIR /content/checkpoints \
    --USE_WANDB 1 --WANDB_PROJECT mtkd4ser-phase1


## 8. Monitor Training (Weights & Biases & TensorBoard)

- **Weights & Biases Dashboard**: Check your live training runs, confusion matrices, and model artifacts at [wandb.ai](https://wandb.ai).
- **TensorBoard**: Optionally visualize local logs via the embedded TensorBoard widget below.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints/tb